# col-check — distributed Stage 2

In [ ]:
from pathlib import Path
import json, os, shlex, subprocess, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
CONFIG = ROOT / 'examples/e2e/config.json'
RUN_S3 = os.getenv('DTRACK_RUN_S3', 's3://dtrack-e2e-work/dtrack/v1/runs')
RUN_ID = os.getenv('DTRACK_RUN_ID', 'nb-distributed')  # Same ID as row-check and the Right Job.
PAIR = 'orders'
WORK = ROOT / '.dtrack-work/col_check'
EXECUTE = False

case = json.loads(CONFIG.read_text())['pairs'][PAIR]
{'vintage': case['vintage'], 'key': case['key'], 'sample': case['sample'], 'columns': case['col_map']}

In [ ]:
def dtrack(*args, execute=EXECUTE):
    command = [sys.executable, '-m', 'dtrack', *map(str, args)]
    print('$', shlex.join(command))
    return subprocess.run(command, cwd=ROOT, text=True, check=execute) if execute else command

# Local/operator producer waits for row_check/_CLOSED, samples Left in-source, then publishes units.
col_command = dtrack(
    'col-check', '--config', CONFIG, '--run-s3', RUN_S3, '--run-id', RUN_ID,
    '--pair', PAIR, '--work-dir', WORK, '--pull-workers', '1', '--stat-workers', '2'
)
col_command

In [ ]:
# The Databricks Job runs concurrently: SparkTarget samples Right, compares units, builds report/.
# Asset Bundle/Workspace Job = distributed mode. Databricks SQL Warehouse = local mode only.
distributed_contract = {
    'local': ['dtrack row-check', 'dtrack col-check'],
    'databricks': ['src/dtrack_right/notebook.py', 'SparkTarget', 'compare', 'report'],
    'shared': f'{RUN_S3}/{RUN_ID}',
    'resume': 'rerun with the same run-id; indexed units are skipped',
}
distributed_contract

In [ ]:
# The Right Job publishes the completed report under this exact run prefix.
{'status': f'{RUN_S3}/{RUN_ID}/_STATUS.json', 'report': f'{RUN_S3}/{RUN_ID}/report/'}

In [ ]:
# Unit contract on real CSVs: normalized key hashes select the same records on both sides.
import tempfile
import pandas as pd
from dtrack_left.common.hashing import sample_csv
from dtrack_left.common.stats import compute_csv_statistics

with tempfile.TemporaryDirectory() as directory:
    temp = Path(directory)
    left = pd.read_csv(ROOT / 'examples/e2e/data/left/ORDERS.csv', dtype=object)
    right = pd.read_csv(ROOT / 'examples/e2e/data/right/orders.csv', dtype=object).rename(columns=str.upper)
    common_days = sorted(set(left.BUSINESS_DATE) & set(right.BUSINESS_DATE))
    left[left.BUSINESS_DATE.isin(common_days)].to_csv(temp / 'left.csv', index=False)
    right[right.BUSINESS_DATE.isin(common_days)].to_csv(temp / 'right.csv', index=False)
    options = {'key_columns': ['ORDER_ID'], 'mode': 'fraction', 'rate': 0.25, 'families': ['numeric']}
    left_sample = pd.read_csv(sample_csv(temp / 'left.csv', **options), dtype=object)
    right_sample = pd.read_csv(sample_csv(temp / 'right.csv', **options), dtype=object)
    clean_left = left_sample[left_sample.BUSINESS_DATE != '2026-01-15']
    clean_right = right_sample[right_sample.BUSINESS_DATE != '2026-01-15']
    assert set(clean_left.ORDER_ID) == set(clean_right.ORDER_ID)
    jan = lambda frame: frame[frame.BUSINESS_DATE.str.startswith('2026-01') & ~frame.BUSINESS_DATE.isin({'2026-01-15', '2026-01-31'})]
    jan(left).to_csv(temp / 'left_jan.csv', index=False)
    jan(right).to_csv(temp / 'right_jan.csv', index=False)
    left_stats = compute_csv_statistics(temp / 'left_jan.csv', ['STATUS', 'REGION', 'AMOUNT'])
    right_stats = compute_csv_statistics(temp / 'right_jan.csv', ['STATUS', 'REGION', 'AMOUNT'])
    assert left_stats['STATUS']['n_unique'] == 4 and right_stats['STATUS']['n_unique'] == 8
    assert left_stats['REGION'] == right_stats['REGION'] and left_stats['AMOUNT'] == right_stats['AMOUNT']
{'sampled_left': len(left_sample), 'sampled_right': len(right_sample), 'status_drift': True}